In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = "KAGGLE_USERNAME"
os.environ['KAGGLE_KEY'] = "KAGGLE_KEY"

!mkdir -p ~/.kaggle
!echo '{"username":"elyaskhalil","key":"be77f658f24b4ffa50993f0778b8276a"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

The syntax of the command is incorrect.
The system cannot find the path specified.
'chmod' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
!kaggle datasets download -d techsash/waste-classification-data -p /content/waste_project --unzip

'kaggle' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
import shutil

src = "/content/waste_project/DATASET"
dst = "/content/dataset"

if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.move(src, dst)

os.rename("/content/dataset/TRAIN", "/content/dataset/train")
os.rename("/content/dataset/TEST", "/content/dataset/test")

!ls /content/dataset

FileNotFoundError: [WinError 3] The system cannot find the path specified

In [ ]:
import random

train_dir = "/content/dataset/train"
val_dir = "/content/dataset/val"

split_ratio = 0.15
for cls in os.listdir(train_dir):
    cls_train = os.path.join(train_dir, cls)
    cls_val = os.path.join(val_dir, cls)
    os.makedirs(cls_val, exist_ok=True)

    imgs = os.listdir(cls_train)
    random.shuffle(imgs)
    n_val = int(len(imgs) * split_ratio)
    val_imgs = imgs[:n_val]

    for img in val_imgs:
        shutil.move(os.path.join(cls_train, img), os.path.join(cls_val, img))

!ls /content/dataset

test  train  val


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (160,160)
BATCH = 64
DATA_DIR = "/content/dataset"

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
).flow_from_directory(
    DATA_DIR + "/train",
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode='categorical'
)

val_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    DATA_DIR + "/val",
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode='categorical',
    shuffle=False
)

test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    DATA_DIR + "/test",
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode='categorical',
    shuffle=False
)

print(train_gen.class_indices)

Found 19181 images belonging to 2 classes.
Found 3383 images belonging to 2 classes.
Found 2513 images belonging to 2 classes.
{'O': 0, 'R': 1}


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(160,160,3))
base_model.trainable = False  # freeze pretrained layers

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
preds = Dense(2, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=preds)

model.compile(optimizer=Adam(learning_rate=0.0005),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
epochs = 6
history = model.fit(train_gen,
                    validation_data=val_gen,
                    epochs=epochs)

Epoch 1/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 165s 490ms/step - accuracy: 0.7861 - loss: 0.4649 - val_accuracy: 0.9113 - val_loss: 0.2260
Epoch 2/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 121s 405ms/step - accuracy: 0.8992 - loss: 0.2528 - val_accuracy: 0.9234 - val_loss: 0.2041
Epoch 3/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 123s 411ms/step - accuracy: 0.9125 - loss: 0.2285 - val_accuracy: 0.9261 - val_loss: 0.1983
Epoch 4/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 121s 404ms/step - accuracy: 0.9197 - loss: 0.2098 - val_accuracy: 0.9228 - val_loss: 0.2121
Epoch 5/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 120s 401ms/step - accuracy: 0.9157 - loss: 0.2156 - val_accuracy: 0.9249 - val_loss: 0.2006
Epoch 6/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 119s 398ms/step - accuracy: 0.9216 - loss: 0.1987 - val_accuracy: 0.9255 - val_loss: 0.1936
Epoch 7/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 119s 396ms/step - accuracy: 0.9181 - loss: 0.2079 - val_accuracy: 0.9308 - val_loss: 0.1905
Epoch 8/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 118s 393ms/step - accuracy: 0.9199 -

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

fine_tune_epochs = 3
model.fit(train_gen,
          validation_data=val_gen,
          epochs=fine_tune_epochs)

Epoch 1/3
300/300 ━━━━━━━━━━━━━━━━━━━━ 163s 469ms/step - accuracy: 0.8999 - loss: 0.2596 - val_accuracy: 0.9166 - val_loss: 0.2476
Epoch 2/3
300/300 ━━━━━━━━━━━━━━━━━━━━ 124s 412ms/step - accuracy: 0.9133 - loss: 0.2197 - val_accuracy: 0.9294 - val_loss: 0.2087
Epoch 3/3
300/300 ━━━━━━━━━━━━━━━━━━━━ 124s 413ms/step - accuracy: 0.9316 - loss: 0.1890 - val_accuracy: 0.9279 - val_loss: 0.2154


In [ ]:
import cv2
import numpy as np

def predict_image(path, model, labels, img_size=(160,160)):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, img_size)
    x = np.expand_dims(img.astype('float32') / 255.0, 0)
    pred = model.predict(x)[0]
    idx = int(np.argmax(pred))
    return labels[idx], float(pred[idx])

In [ ]:
import os

test_dir = "/content/dataset/test/R"  # or "/content/dataset/test/O"
sample_img = os.path.join(test_dir, os.listdir(test_dir)[0])
print("Sample image:", sample_img)

Sample image: /content/dataset/test/O/O_13163.jpg


In [ ]:
labels = {0: 'O', 1: 'R'}

In [ ]:
label, confidence = predict_image(sample_img, model, labels)
print("Predicted:", label)
print("Confidence:", round(confidence, 3))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
Predicted: O
Confidence: 1.0


In [ ]:
import os

true_label = os.path.basename(os.path.dirname(sample_img))
print("True label:", true_label)
print("Prediction correct:", label == true_label)

True label: O
Prediction correct: True


In [ ]:
import random

test_folder = "/content/dataset/test/O"
samples = random.sample(os.listdir(test_folder), 10)

for file in samples:
    img_path = os.path.join(test_folder, file)
    label, conf = predict_image(img_path, model, labels)
    true_label = os.path.basename(os.path.dirname(img_path))
    print(f"File: {file}")
    print(f"True: {true_label}, Predicted: {label}, Confidence: {conf:.2f}")
    print()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
File: O_13388.jpg
True: O, Predicted: O, Confidence: 1.00

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
File: O_12946.jpg
True: O, Predicted: O, Confidence: 0.99

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
File: O_12938.jpg
True: O, Predicted: O, Confidence: 0.99

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
File: O_13771.jpg
True: O, Predicted: O, Confidence: 1.00

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
File: O_13502.jpg
True: O, Predicted: O, Confidence: 1.00

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
File: O_13111.jpg
True: O, Predicted: O, Confidence: 1.00

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
File: O_13499.jpg
True: O, Predicted: O, Confidence: 1.00

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
File: O_12853.jpg
True: O, Predicted: O, Confidence: 0.95

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
File: O_12583.jpg
True: O, Predicted: O, Confidence: 1.00

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
File: O_13460.jpg
True: O, Predicted: O, Confidence: 0.59

